# Mandatory Practical Work 01 – CNNs on iCoSimal V3 Dataset

**MSE FTP Deep Learning – 17/03/2026**

---

**Group Members:**
- Student 1: [Name, Surname]
- Student 2: [Name, Surname]
- Student 3: [Name, Surname]

---

## Table of Contents

1. [Setup & Data Loading](#1-setup)
2. [Data Exploration](#2-exploration)
3. [Objective (a): Progressive CNN Architectures](#3-progressive-cnn)
4. [Objective (b): Hyperparameter Tuning](#4-hyperparameter-tuning)
5. [Objective (c): Overfitting & Underfitting](#5-overfit-underfit)
6. [Objective (d): Regularization Techniques](#6-regularization)
7. [Objective (e): Optimization Algorithms](#7-optimizers)
8. [Optional: Data Augmentation](#8-data-augmentation)
9. [Optional: Transfer Learning](#9-transfer-learning)
10. [Optional: Error Analysis & Grad-CAM](#10-error-analysis)
11. [Summary & Conclusions](#11-summary)

---
## 1. Setup & Data Loading <a id='1-setup'></a>

In [1]:
import os
import copy
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import torchvision
from torchvision import datasets, transforms, models

from sklearn.metrics import confusion_matrix, classification_report

# Reproducibility
SEED = 4869
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

Using device: cpu


In [2]:
# ============================================================
# CONFIGURATION – adjust paths and image size here
# ============================================================
DATA_DIR = 'icosimal_img_class_03/data_uniform_224_224_sets'  # <-- adjust if needed
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR   = os.path.join(DATA_DIR, 'validate')

# Start with 64x64 for fast iteration; increase to 128 or 224 for final runs
IMG_SIZE = 64
BATCH_SIZE = 64
NUM_CLASSES = 10
NUM_WORKERS = 4

CLASS_NAMES = sorted(os.listdir(TRAIN_DIR))
print(f'Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}')

FileNotFoundError: [Errno 2] No such file or directory: 'icosimal_img_class_03/data_uniform_224_224_sets/train'

In [3]:
# ============================================================
# Data transforms (basic – no augmentation yet)
# ============================================================
def get_transforms(img_size, augment=False):
    """Return train and validation transforms."""
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
    if augment:
        train_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            transforms.ToTensor(),
            normalize,
        ])
    else:
        train_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            normalize,
        ])
    val_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        normalize,
    ])
    return train_tf, val_tf


def get_dataloaders(img_size, batch_size, augment=False):
    """Create train and validation dataloaders."""
    train_tf, val_tf = get_transforms(img_size, augment)
    train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
    val_ds   = datasets.ImageFolder(VAL_DIR,   transform=val_tf)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, train_ds, val_ds


train_loader, val_loader, train_ds, val_ds = get_dataloaders(IMG_SIZE, BATCH_SIZE)
print(f'Training samples:   {len(train_ds)}')
print(f'Validation samples: {len(val_ds)}')

FileNotFoundError: [Errno 2] No such file or directory: 'icosimal_img_class_03/data_uniform_224_224_sets/train'

---
## 2. Data Exploration <a id='2-exploration'></a>

In [ ]:
# Show class distribution
from collections import Counter
train_labels = [label for _, label in train_ds.samples]
val_labels   = [label for _, label in val_ds.samples]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, labels, title in zip(axes, [train_labels, val_labels], ['Train', 'Validation']):
    counts = Counter(labels)
    ax.bar([CLASS_NAMES[i] for i in sorted(counts)], [counts[i] for i in sorted(counts)])
    ax.set_title(f'{title} Set Distribution')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Show sample images
def show_batch(loader, n=16):
    images, labels = next(iter(loader))
    fig, axes = plt.subplots(2, 8, figsize=(16, 4))
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    for i, ax in enumerate(axes.flat):
        if i >= n:
            break
        img = images[i] * std + mean  # denormalize
        ax.imshow(img.permute(1, 2, 0).clamp(0, 1).numpy())
        ax.set_title(CLASS_NAMES[labels[i]], fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_batch(train_loader)

---
## Training Utilities

We define a reusable training loop and evaluation helpers used throughout the notebook.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)
    return running_loss / total, correct / total


def train_model(model, train_loader, val_loader, criterion, optimizer,
                scheduler=None, epochs=20, device=DEVICE, early_stop_patience=None):
    """Full training loop with history tracking and optional early stopping."""
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc     = evaluate(model, val_loader, criterion, device)
        if scheduler is not None:
            scheduler.step()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        elapsed = time.time() - t0
        print(f'Epoch {epoch:3d}/{epochs} | '
              f'Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | '
              f'Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f} | '
              f'{elapsed:.1f}s')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if early_stop_patience and patience_counter >= early_stop_patience:
            print(f'Early stopping at epoch {epoch}')
            break

    model.load_state_dict(best_model_wts)
    print(f'\nBest validation accuracy: {best_val_acc:.4f}')
    return model, history


def plot_history(history, title=''):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    ax1.plot(history['train_loss'], label='Train')
    ax1.plot(history['val_loss'],   label='Val')
    ax1.set_title(f'Loss {title}')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()
    ax2.plot(history['train_acc'], label='Train')
    ax2.plot(history['val_acc'],   label='Val')
    ax2.set_title(f'Accuracy {title}')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend()
    plt.tight_layout(); plt.show()

---
## 3. Objective (a): Progressive CNN Architectures <a id='3-progressive-cnn'></a>

We start with a very simple 1-conv-layer CNN and progressively add depth to demonstrate the benefit of deeper networks.

In [ ]:
# ============================================================
# Model A: Minimal CNN – 1 conv layer
# ============================================================
class SimpleCNN_1(nn.Module):
    """Minimal CNN: Conv -> Pool -> FC."""
    def __init__(self, img_size=64, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # img_size/2
        )
        feat_size = img_size // 2
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * feat_size * feat_size, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ============================================================
# Model B: 3 conv layers
# ============================================================
class SimpleCNN_3(nn.Module):
    """3 conv blocks: Conv-BN-ReLU-Pool."""
    def __init__(self, img_size=64, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,  32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128, 3, padding=1), nn.BatchNorm2d(128),nn.ReLU(), nn.MaxPool2d(2),
        )
        feat_size = img_size // 8
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * feat_size * feat_size, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ============================================================
# Model C: 5 conv layers
# ============================================================
class SimpleCNN_5(nn.Module):
    """5 conv blocks with increasing filters."""
    def __init__(self, img_size=64, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,   32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,  64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256,512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


print('Model architectures defined.')

In [ ]:
# Count parameters
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for name, cls in [('SimpleCNN_1', SimpleCNN_1), ('SimpleCNN_3', SimpleCNN_3), ('SimpleCNN_5', SimpleCNN_5)]:
    m = cls(img_size=IMG_SIZE, num_classes=NUM_CLASSES)
    print(f'{name}: {count_params(m):,} trainable parameters')

In [ ]:
# Train all three architectures
EPOCHS_ARCH = 15
results_arch = {}

for name, ModelClass in [('1-layer CNN', SimpleCNN_1),
                          ('3-layer CNN', SimpleCNN_3),
                          ('5-layer CNN', SimpleCNN_5)]:
    print(f'\n{"="*60}')
    print(f'Training: {name}')
    print(f'{"="*60}')
    model = ModelClass(img_size=IMG_SIZE, num_classes=NUM_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    model, history = train_model(model, train_loader, val_loader, criterion, optimizer,
                                  epochs=EPOCHS_ARCH)
    results_arch[name] = history
    plot_history(history, title=f'– {name}')

In [ ]:
# Compare architectures
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for name, h in results_arch.items():
    ax1.plot(h['val_loss'], label=name)
    ax2.plot(h['val_acc'],  label=name)
ax1.set_title('Validation Loss by Architecture'); ax1.legend(); ax1.set_xlabel('Epoch')
ax2.set_title('Validation Accuracy by Architecture'); ax2.legend(); ax2.set_xlabel('Epoch')
plt.tight_layout(); plt.show()

print('\n--- Final validation accuracy ---')
for name, h in results_arch.items():
    print(f'{name}: {max(h["val_acc"]):.4f}')

**Analysis (a):**  
The results show that increasing the number of convolutional layers generally improves validation accuracy. The 1-layer CNN has limited capacity to learn complex features, while the 3- and 5-layer CNNs can capture hierarchical features (edges → textures → parts → objects), leading to better performance.

---
## 4. Objective (b): Hyperparameter Tuning <a id='4-hyperparameter-tuning'></a>

We systematically vary key hyperparameters (learning rate, batch size) using the 5-layer CNN and show their impact on training dynamics and final performance.

In [ ]:
# ============================================================
# 4.1 Learning rate sweep
# ============================================================
EPOCHS_HP = 15
lr_results = {}

for lr in [1e-4, 5e-4, 1e-3, 5e-3, 1e-2]:
    print(f'\n--- LR = {lr} ---')
    model = SimpleCNN_5(img_size=IMG_SIZE, num_classes=NUM_CLASSES).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    model, history = train_model(model, train_loader, val_loader, criterion, optimizer,
                                  epochs=EPOCHS_HP)
    lr_results[f'lr={lr}'] = history

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for name, h in lr_results.items():
    ax1.plot(h['val_loss'], label=name)
    ax2.plot(h['val_acc'],  label=name)
ax1.set_title('Validation Loss vs Learning Rate'); ax1.legend(); ax1.set_xlabel('Epoch')
ax2.set_title('Validation Accuracy vs Learning Rate'); ax2.legend(); ax2.set_xlabel('Epoch')
plt.tight_layout(); plt.show()

print('\n--- Best validation accuracy per LR ---')
for name, h in lr_results.items():
    print(f'{name}: {max(h["val_acc"]):.4f}')

In [ ]:
# ============================================================
# 4.2 Batch size sweep
# ============================================================
bs_results = {}
BEST_LR = 1e-3  # <-- set this to the best LR found above

for bs in [16, 32, 64, 128, 256]:
    print(f'\n--- Batch Size = {bs} ---')
    tl, vl, _, _ = get_dataloaders(IMG_SIZE, bs)
    model = SimpleCNN_5(img_size=IMG_SIZE, num_classes=NUM_CLASSES).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=BEST_LR)
    criterion = nn.CrossEntropyLoss()
    model, history = train_model(model, tl, vl, criterion, optimizer, epochs=EPOCHS_HP)
    bs_results[f'bs={bs}'] = history

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for name, h in bs_results.items():
    ax1.plot(h['val_loss'], label=name)
    ax2.plot(h['val_acc'],  label=name)
ax1.set_title('Validation Loss vs Batch Size'); ax1.legend(); ax1.set_xlabel('Epoch')
ax2.set_title('Validation Accuracy vs Batch Size'); ax2.legend(); ax2.set_xlabel('Epoch')
plt.tight_layout(); plt.show()

print('\n--- Best validation accuracy per Batch Size ---')
for name, h in bs_results.items():
    print(f'{name}: {max(h["val_acc"]):.4f}')

**Analysis (b):**  
- **Learning rate** is the most critical hyperparameter. Too low → slow convergence; too high → unstable training / divergence.
- **Batch size** affects the noise in gradient estimates: smaller batches provide implicit regularization but slower wall-clock time; larger batches converge smoother but may generalize worse without LR scaling.

---
## 5. Objective (c): Overfitting & Underfitting <a id='5-overfit-underfit'></a>

We deliberately create conditions that lead to overfitting and underfitting.

In [ ]:
# ============================================================
# 5.1 UNDERFITTING: tiny model + few epochs + high LR
# ============================================================
print('=== Underfitting Demo ===')
model_under = SimpleCNN_1(img_size=IMG_SIZE, num_classes=NUM_CLASSES).to(DEVICE)
optimizer = optim.SGD(model_under.parameters(), lr=0.0001)  # very low LR + weak model
criterion = nn.CrossEntropyLoss()
model_under, hist_under = train_model(model_under, train_loader, val_loader, criterion,
                                       optimizer, epochs=10)
plot_history(hist_under, title='– Underfitting (tiny model, low LR)')

In [ ]:
# ============================================================
# 5.2 OVERFITTING: large model on small subset, no regularization, many epochs
# ============================================================
print('=== Overfitting Demo ===')

# Use only 500 training samples to make overfitting easy
subset_indices = list(range(500))
subset_ds = Subset(train_ds, subset_indices)
subset_loader = DataLoader(subset_ds, batch_size=32, shuffle=True, num_workers=NUM_WORKERS)

model_over = SimpleCNN_5(img_size=IMG_SIZE, num_classes=NUM_CLASSES).to(DEVICE)
optimizer = optim.Adam(model_over.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
model_over, hist_over = train_model(model_over, subset_loader, val_loader, criterion,
                                     optimizer, epochs=40)
plot_history(hist_over, title='– Overfitting (large model, small data, no reg.)')

**Analysis (c):**
- **Underfitting**: Both train and val accuracy are low. The model lacks capacity (only 1 conv layer) and the learning rate is too small to learn useful features in few epochs.
- **Overfitting**: Train accuracy reaches ~100% while val accuracy plateaus much lower. The large model memorizes the small training set instead of learning generalizable features. The gap between train and val loss keeps widening.

---
## 6. Objective (d): Regularization Techniques <a id='6-regularization'></a>

We apply dropout, weight decay (L2), and data augmentation to the overfitting scenario to show their effect.

In [ ]:
class CNN5_Regularized(nn.Module):
    """5-layer CNN with dropout after FC layers."""
    def __init__(self, img_size=64, num_classes=10, dropout=0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,   32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,  64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256,512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
# ============================================================
# 6.1 Dropout
# ============================================================
print('=== Regularization: Dropout 0.5 ===')
model_drop = CNN5_Regularized(img_size=IMG_SIZE, num_classes=NUM_CLASSES, dropout=0.5).to(DEVICE)
optimizer = optim.Adam(model_drop.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
model_drop, hist_drop = train_model(model_drop, subset_loader, val_loader, criterion,
                                     optimizer, epochs=40)
plot_history(hist_drop, title='– With Dropout 0.5')

In [ ]:
# ============================================================
# 6.2 Weight Decay (L2 regularization)
# ============================================================
print('=== Regularization: Weight Decay 1e-3 ===')
model_wd = SimpleCNN_5(img_size=IMG_SIZE, num_classes=NUM_CLASSES).to(DEVICE)
optimizer = optim.Adam(model_wd.parameters(), lr=1e-3, weight_decay=1e-3)
criterion = nn.CrossEntropyLoss()
model_wd, hist_wd = train_model(model_wd, subset_loader, val_loader, criterion,
                                 optimizer, epochs=40)
plot_history(hist_wd, title='– With Weight Decay 1e-3')

In [ ]:
# ============================================================
# 6.3 Dropout + Weight Decay combined
# ============================================================
print('=== Regularization: Dropout + Weight Decay ===')
model_both = CNN5_Regularized(img_size=IMG_SIZE, num_classes=NUM_CLASSES, dropout=0.5).to(DEVICE)
optimizer = optim.Adam(model_both.parameters(), lr=1e-3, weight_decay=1e-3)
criterion = nn.CrossEntropyLoss()
model_both, hist_both = train_model(model_both, subset_loader, val_loader, criterion,
                                     optimizer, epochs=40)
plot_history(hist_both, title='– Dropout + Weight Decay')

In [ ]:
# Compare regularization approaches
reg_results = {
    'No reg (overfit)': hist_over,
    'Dropout 0.5': hist_drop,
    'Weight Decay 1e-3': hist_wd,
    'Dropout + WD': hist_both,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, h in reg_results.items():
    axes[0].plot(h['val_loss'], label=name)
    axes[1].plot(h['val_acc'],  label=name)
axes[0].set_title('Val Loss – Regularization Comparison'); axes[0].legend()
axes[1].set_title('Val Accuracy – Regularization Comparison'); axes[1].legend()
plt.tight_layout(); plt.show()

print('\n--- Best val accuracy ---')
for name, h in reg_results.items():
    print(f'{name}: {max(h["val_acc"]):.4f}')

**Analysis (d):**  
Regularization techniques reduce the train-val gap by constraining model complexity. Dropout randomly silences neurons, forcing the network to learn redundant representations. Weight decay penalizes large weights, encouraging simpler solutions. Combining both typically provides the best generalization.

---
## 7. Objective (e): Optimization Algorithms <a id='7-optimizers'></a>

We compare SGD (with momentum), Adam, RMSprop, and AdamW on the full training set.

In [ ]:
EPOCHS_OPT = 20
opt_results = {}

optimizer_configs = {
    'SGD (lr=0.01, mom=0.9)': lambda p: optim.SGD(p, lr=0.01, momentum=0.9),
    'SGD + Nesterov':         lambda p: optim.SGD(p, lr=0.01, momentum=0.9, nesterov=True),
    'Adam (lr=1e-3)':         lambda p: optim.Adam(p, lr=1e-3),
    'AdamW (lr=1e-3)':        lambda p: optim.AdamW(p, lr=1e-3, weight_decay=1e-2),
    'RMSprop (lr=1e-3)':      lambda p: optim.RMSprop(p, lr=1e-3),
}

for opt_name, opt_fn in optimizer_configs.items():
    print(f'\n{"="*60}')
    print(f'Optimizer: {opt_name}')
    print(f'{"="*60}')
    model = SimpleCNN_5(img_size=IMG_SIZE, num_classes=NUM_CLASSES).to(DEVICE)
    optimizer = opt_fn(model.parameters())
    criterion = nn.CrossEntropyLoss()
    model, history = train_model(model, train_loader, val_loader, criterion, optimizer,
                                  epochs=EPOCHS_OPT)
    opt_results[opt_name] = history

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
for name, h in opt_results.items():
    ax1.plot(h['val_loss'], label=name)
    ax2.plot(h['val_acc'],  label=name)
ax1.set_title('Validation Loss – Optimizer Comparison'); ax1.legend(fontsize=8)
ax2.set_title('Validation Accuracy – Optimizer Comparison'); ax2.legend(fontsize=8)
ax1.set_xlabel('Epoch'); ax2.set_xlabel('Epoch')
plt.tight_layout(); plt.show()

print('\n--- Best validation accuracy per optimizer ---')
for name, h in opt_results.items():
    print(f'{name}: {max(h["val_acc"]):.4f}')

**Analysis (e):**  
- **SGD** converges slowly but can reach good minima with proper tuning and scheduling.
- **Adam/AdamW** converge much faster thanks to adaptive learning rates. AdamW decouples weight decay from gradient updates, often leading to better generalization.
- **RMSprop** is similar to Adam but lacks momentum correction; performance is usually between SGD and Adam.

---
## 8. Optional: Data Augmentation <a id='8-data-augmentation'></a>

We compare training with and without data augmentation on the full dataset.

In [ ]:
# Visualize augmented samples
train_tf_aug, _ = get_transforms(IMG_SIZE, augment=True)
aug_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tf_aug)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
# Show same image augmented 8 times
idx = 0
for ax in axes.flat:
    img, lbl = aug_ds[idx]
    img = img * std + mean
    ax.imshow(img.permute(1,2,0).clamp(0,1).numpy())
    ax.set_title(CLASS_NAMES[lbl], fontsize=9)
    ax.axis('off')
plt.suptitle('Same image with different augmentations', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Train with augmentation
EPOCHS_AUG = 25

print('=== Without Augmentation ===')
tl_no_aug, vl_no_aug, _, _ = get_dataloaders(IMG_SIZE, BATCH_SIZE, augment=False)
model_no_aug = CNN5_Regularized(img_size=IMG_SIZE, num_classes=NUM_CLASSES, dropout=0.3).to(DEVICE)
optimizer = optim.AdamW(model_no_aug.parameters(), lr=1e-3, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()
model_no_aug, hist_no_aug = train_model(model_no_aug, tl_no_aug, vl_no_aug, criterion,
                                         optimizer, epochs=EPOCHS_AUG)

print('\n=== With Augmentation ===')
tl_aug, vl_aug, _, _ = get_dataloaders(IMG_SIZE, BATCH_SIZE, augment=True)
model_aug = CNN5_Regularized(img_size=IMG_SIZE, num_classes=NUM_CLASSES, dropout=0.3).to(DEVICE)
optimizer = optim.AdamW(model_aug.parameters(), lr=1e-3, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()
model_aug, hist_aug = train_model(model_aug, tl_aug, vl_aug, criterion,
                                   optimizer, epochs=EPOCHS_AUG)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for name, h in [('No Aug', hist_no_aug), ('With Aug', hist_aug)]:
    ax1.plot(h['val_loss'], label=name)
    ax2.plot(h['val_acc'],  label=name)
ax1.set_title('Val Loss – Augmentation'); ax1.legend(); ax1.set_xlabel('Epoch')
ax2.set_title('Val Accuracy – Augmentation'); ax2.legend(); ax2.set_xlabel('Epoch')
plt.tight_layout(); plt.show()

print(f'Best val acc without augmentation: {max(hist_no_aug["val_acc"]):.4f}')
print(f'Best val acc with augmentation:    {max(hist_aug["val_acc"]):.4f}')

**Analysis (Data Augmentation):**  
Data augmentation acts as an implicit regularizer by presenting different views of the same image. Training loss is higher (harder task) but the model generalizes better, as seen in improved validation accuracy and reduced train-val gap.

---
## 9. Optional: Transfer Learning <a id='9-transfer-learning'></a>

We fine-tune a pretrained ResNet-18 and compare it with our best model trained from scratch.

In [ ]:
# Use 224x224 for transfer learning (ImageNet pretrained models expect this)
TL_IMG_SIZE = 128  # Use 128 for speed; switch to 224 for best results
tl_train_loader, tl_val_loader, _, _ = get_dataloaders(TL_IMG_SIZE, BATCH_SIZE, augment=True)

In [ ]:
# ============================================================
# 9.1 Feature extraction (freeze backbone)
# ============================================================
print('=== Transfer Learning: Feature Extraction (frozen backbone) ===')
model_fe = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
# Freeze all parameters
for param in model_fe.parameters():
    param.requires_grad = False
# Replace classifier head
model_fe.fc = nn.Sequential(
    nn.Linear(model_fe.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, NUM_CLASSES),
)
model_fe = model_fe.to(DEVICE)
print(f'Trainable params: {count_params(model_fe):,}')

optimizer = optim.Adam(model_fe.fc.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
model_fe, hist_fe = train_model(model_fe, tl_train_loader, tl_val_loader, criterion,
                                 optimizer, epochs=15)
plot_history(hist_fe, title='– ResNet18 Feature Extraction')

In [ ]:
# ============================================================
# 9.2 Fine-tuning (unfreeze last layers)
# ============================================================
print('=== Transfer Learning: Fine-Tuning ===')
model_ft = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model_ft.fc = nn.Sequential(
    nn.Linear(model_ft.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, NUM_CLASSES),
)

# Freeze early layers, unfreeze layer3, layer4, and fc
for name, param in model_ft.named_parameters():
    if 'layer3' not in name and 'layer4' not in name and 'fc' not in name:
        param.requires_grad = False

model_ft = model_ft.to(DEVICE)
print(f'Trainable params: {count_params(model_ft):,}')

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model_ft.parameters()),
                        lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
criterion = nn.CrossEntropyLoss()
model_ft, hist_ft = train_model(model_ft, tl_train_loader, tl_val_loader, criterion,
                                 optimizer, scheduler=scheduler, epochs=20)
plot_history(hist_ft, title='– ResNet18 Fine-Tuning')

In [ ]:
print('\n--- Transfer Learning Summary ---')
print(f'Feature Extraction best val acc: {max(hist_fe["val_acc"]):.4f}')
print(f'Fine-Tuning best val acc:        {max(hist_ft["val_acc"]):.4f}')

**Analysis (Transfer Learning):**  
Pretrained models already encode rich visual features from ImageNet. Even with a frozen backbone, the feature extraction approach often outperforms training from scratch. Fine-tuning the later layers further adapts these features to the specific animal classes in iCoSimal V3, yielding the best results.

---
## 10. Optional: Error Analysis & Grad-CAM <a id='10-error-analysis'></a>

We analyze misclassifications of the best model and use Grad-CAM to visualize what the model focuses on.

In [ ]:
# ============================================================
# 10.1 Confusion Matrix & Classification Report
# ============================================================
best_model = model_ft  # use the fine-tuned ResNet18
best_model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tl_val_loader:
        images = images.to(DEVICE)
        outputs = best_model(images)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Confusion Matrix – Best Model')
plt.tight_layout(); plt.show()

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

In [ ]:
# ============================================================
# 10.2 Show misclassified examples
# ============================================================
misclassified_idx = np.where(all_preds != all_labels)[0]
print(f'Total misclassified: {len(misclassified_idx)} / {len(all_labels)}')

# Show 16 random misclassifications
_, val_tf = get_transforms(TL_IMG_SIZE)
val_ds_display = datasets.ImageFolder(VAL_DIR, transform=val_tf)

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

sample_mis = np.random.choice(misclassified_idx, min(16, len(misclassified_idx)), replace=False)
for i, ax in enumerate(axes.flat):
    if i >= len(sample_mis):
        ax.axis('off'); continue
    idx = sample_mis[i]
    img, true_label = val_ds_display[idx]
    img_show = (img * std + mean).permute(1,2,0).clamp(0,1).numpy()
    ax.imshow(img_show)
    ax.set_title(f'T:{CLASS_NAMES[true_label]}\nP:{CLASS_NAMES[all_preds[idx]]}', fontsize=8,
                 color='red')
    ax.axis('off')
plt.suptitle('Misclassified Examples (True vs Predicted)', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 10.3 Grad-CAM Visualization
# ============================================================
class GradCAM:
    """Simple Grad-CAM implementation for ResNet."""
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        output = self.model(input_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1).item()

        self.model.zero_grad()
        output[0, target_class].backward()

        weights = self.gradients.mean(dim=[2, 3], keepdim=True)  # GAP of gradients
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, target_class


# Apply Grad-CAM on the fine-tuned ResNet18
grad_cam = GradCAM(best_model, best_model.layer4[-1])

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
indices = np.random.choice(len(val_ds_display), 6, replace=False)

for i, idx in enumerate(indices):
    img, true_label = val_ds_display[idx]
    input_tensor = img.unsqueeze(0).to(DEVICE)

    cam, pred_class = grad_cam.generate(input_tensor)

    img_show = (img * std + mean).permute(1,2,0).clamp(0,1).numpy()

    # Original image
    axes[0, i].imshow(img_show)
    axes[0, i].set_title(f'True: {CLASS_NAMES[true_label]}', fontsize=9)
    axes[0, i].axis('off')

    # Grad-CAM overlay
    axes[1, i].imshow(img_show)
    axes[1, i].imshow(cam, cmap='jet', alpha=0.5)
    axes[1, i].set_title(f'Pred: {CLASS_NAMES[pred_class]}', fontsize=9)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=11)
axes[1, 0].set_ylabel('Grad-CAM', fontsize=11)
plt.suptitle('Grad-CAM Visualizations', fontsize=13)
plt.tight_layout(); plt.show()

**Analysis (Error Analysis & Grad-CAM):**  
- The confusion matrix reveals which class pairs are most commonly confused (e.g., cat/rabbit, cow/horse).
- Grad-CAM shows that the model typically focuses on the animal's body and distinctive features (ears, stripes, tusks) rather than the background, confirming it has learned meaningful features.

---
## 11. Summary & Conclusions <a id='11-summary'></a>

In [ ]:
# ============================================================
# Final results summary table
# ============================================================
summary_data = []

# Collect all results
all_experiments = {}
all_experiments.update(results_arch)
all_experiments['Best optimizer'] = opt_results[max(opt_results, key=lambda k: max(opt_results[k]['val_acc']))]
all_experiments['With augmentation'] = hist_aug
all_experiments['ResNet18 (feature extraction)'] = hist_fe
all_experiments['ResNet18 (fine-tuned)'] = hist_ft

print(f'{"Experiment":<40} {"Best Val Acc":>12} {"Final Train Acc":>15}')
print('-' * 70)
for name, h in all_experiments.items():
    print(f'{name:<40} {max(h["val_acc"]):>12.4f} {h["train_acc"][-1]:>15.4f}')

### Key Takeaways

1. **Depth matters (Obj. a):** Progressively deeper CNNs capture increasingly abstract features, leading to better accuracy. However, depth alone is not sufficient — proper regularization is needed.

2. **Hyperparameter tuning is crucial (Obj. b):** The learning rate has the largest impact on performance. Batch size affects training dynamics and implicit regularization.

3. **Overfitting vs. Underfitting (Obj. c):** Overfitting occurs when model capacity exceeds data complexity (large model + small data); underfitting when the model lacks capacity or training is insufficient.

4. **Regularization works (Obj. d):** Dropout and weight decay reduce overfitting by constraining model complexity. Combining multiple regularization techniques often yields the best generalization.

5. **Optimizer choice matters (Obj. e):** Adaptive optimizers (Adam, AdamW) converge faster than SGD. AdamW with proper weight decay often provides the best trade-off between convergence speed and generalization.

6. **Data augmentation (Optional):** Augmentation improves generalization by artificially increasing training data diversity.

7. **Transfer learning (Optional):** Pretrained models dramatically outperform training from scratch, especially with limited data. Fine-tuning the later layers further adapts the model to the target domain.

8. **Error analysis & Grad-CAM (Optional):** Understanding model failures helps identify dataset issues and guides architecture improvements. Grad-CAM confirms the model attends to semantically meaningful regions.